# MagiAttention SP vs FA3 head-parallel (Ulysses)

Multi-GPU check of **MagiAttention sequence parallel** against **FA3 paralleled by heads** (Ulysses-style all2all).

| Setting | Value |
|---------|-------|
| `seqlen` | 4096 |
| GPUs | 2 and 4 |
| Magi | SP / CP along sequence (`magi_attn_flex_key` + `calc_attn`) |
| Baseline | FA3 + all2all head parallel (contiguous seq shard) |
| Masks | `full`, `causal` |
| Heads | `H = 128` (MHA) |
| Head dims | `(dq, dk, dv) ∈ {(192, 192, 128), (192, 192, 192)}` |
| dtype | `bfloat16` |

The notebook launches `magi_sp_vs_fa3_ulysses_worker.py` via `torchrun`, then prints / saves tables under `results/`.

Recommended kernel: `dmikhaylov-k6v_magi` (FA3 + Magi FFA). Magi multi-GPU needs `magi_attn_comm` / `magi_attn_ext` (symlinked from an installed wheel if you develop from source).



In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd

# Resolve repo / experiment dirs whether cwd is repo root or this folder
HERE = Path.cwd().resolve()
CANDIDATES = [HERE, HERE / "exps/attn/multigpu", *HERE.parents]
EXP_DIR = None
REPO = None
for p in CANDIDATES:
    if (p / "magi_sp_vs_fa3_ulysses_worker.py").is_file():
        EXP_DIR = p
        break
if EXP_DIR is None:
    raise FileNotFoundError("Cannot find magi_sp_vs_fa3_ulysses_worker.py; cd to exps/attn/multigpu")
for p in [EXP_DIR, *EXP_DIR.parents]:
    if (p / "magi_attention").is_dir() and (p / "magi_attention" / "__init__.py").is_file():
        REPO = p
        break
assert REPO is not None

RESULTS = EXP_DIR / "results"
RESULTS.mkdir(exist_ok=True)
WORKER = EXP_DIR / "magi_sp_vs_fa3_ulysses_worker.py"

os.environ["PYTHONPATH"] = str(REPO) + (
    os.pathsep + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else ""
)

# If developing from source without an editable install, link compiled extensions
# from a known env that has them (optional; skipped when already present).
EXT_DIR = REPO / "magi_attention"
SO_SRC = Path(
    os.environ.get(
        "MAGI_ATTN_SO_DIR",
        "/home/jovyan/dmikhaylov/envs/k6_img_nf4/lib/python3.13/site-packages/magi_attention",
    )
)
for name in [
    "magi_attn_ext.cpython-313-x86_64-linux-gnu.so",
    "magi_attn_comm.cpython-313-x86_64-linux-gnu.so",
    "flexible_flash_attention_utils_cuda.cpython-313-x86_64-linux-gnu.so",
]:
    dst = EXT_DIR / name
    src = SO_SRC / name
    if not dst.exists() and src.exists():
        dst.symlink_to(src)
        print(f"linked {name}")

PYTHON = sys.executable
TORCHRUN = shutil.which("torchrun") or str(Path(PYTHON).parent / "torchrun")
print("REPO   ", REPO)
print("EXP_DIR", EXP_DIR)
print("python ", PYTHON)
print("torchrun", TORCHRUN)

import torch

print(f"torch {torch.__version__} | gpus={torch.cuda.device_count()}")
import flash_attn_interface  # noqa: F401

print("FA3: ok")
from magi_attention import magi_attn_comm, magi_attn_ext  # noqa: F401

print("Magi extensions: ok")

## Config

`H` must be divisible by the largest CP size (4). Lower `WARMUP` / `ITERS` for a quick smoke.



In [ ]:
SEQLEN = 4096
NUM_HEADS = 128
# (dq, dk, dv) — Magi / FA3 both support dv != dqk (e.g. DeepSeek 192/128)
HEAD_DIMS = [(192, 192, 128), (192, 192, 192)]
HEAD_DIMS_ARG = ";".join(f"{dq},{dk},{dv}" for dq, dk, dv in HEAD_DIMS)
CHUNK_SIZE = 512
DTYPE = "bfloat16"
MASKS = "full,causal"
WORLD_SIZES = [2, 4]
WARMUP = 5
ITERS = 20
MASTER_PORT = int(os.environ.get("MASTER_PORT", "29531"))

assert torch.cuda.device_count() >= max(WORLD_SIZES), (
    f"need >= {max(WORLD_SIZES)} GPUs, got {torch.cuda.device_count()}"
)
for ws in WORLD_SIZES:
    assert NUM_HEADS % ws == 0
    assert SEQLEN % ws == 0
for dq, dk, dv in HEAD_DIMS:
    assert dq == dk, (dq, dk, dv)

print("config ok", f"H={NUM_HEADS}", f"dims={HEAD_DIMS}")



## Run (torchrun)

Each world size writes `results/ws{N}.json`. First Magi call may JIT-compile FFA kernels (can take minutes).

In [ ]:
def run_world_size(ws: int) -> Path:
    out = RESULTS / f"ws{ws}.json"
    cmd = [
        TORCHRUN,
        "--standalone",
        f"--nproc_per_node={ws}",
        f"--master_port={MASTER_PORT + ws}",
        str(WORKER),
        "--out",
        str(out),
        "--seqlen",
        str(SEQLEN),
        "--num-heads",
        str(NUM_HEADS),
        "--head-dims",
        HEAD_DIMS_ARG,
        "--chunk-size",
        str(CHUNK_SIZE),
        "--dtype",
        DTYPE,
        "--masks",
        MASKS,
        "--warmup",
        str(WARMUP),
        "--iters",
        str(ITERS),
    ]
    env = os.environ.copy()
    env["PYTHONPATH"] = str(REPO) + (
        os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else ""
    )
    print("+", " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=str(REPO), env=env, check=False)
    if proc.returncode != 0:
        raise RuntimeError(f"torchrun failed for world_size={ws} (exit {proc.returncode})")
    assert out.is_file(), out
    return out


json_paths = {}
for ws in WORLD_SIZES:
    json_paths[ws] = run_world_size(ws)
    print("wrote", json_paths[ws])



## Tables + save markdown / csv

In [ ]:
payloads = {ws: json.loads(Path(p).read_text()) for ws, p in json_paths.items()}

corr_rows = []
thr_rows = []
for ws, payload in payloads.items():
    for r in payload["results"]:
        c = r["correctness"]
        dims = f"({r['head_dim_q']},{r['head_dim_k']},{r['head_dim_v']})"
        corr_rows.append(
            {
                "world_size": ws,
                "mask": r["mask"],
                "dims": dims,
                "dq": r["head_dim_q"],
                "dk": r["head_dim_k"],
                "dv": r["head_dim_v"],
                **{k: c[k] for k in c if k != "mask"},
            }
        )
        t = r["throughput"]
        thr_rows.append(
            {
                "world_size": ws,
                "mask": r["mask"],
                "dims": dims,
                "dq": r["head_dim_q"],
                "dk": r["head_dim_k"],
                "dv": r["head_dim_v"],
                **t,
            }
        )

corr_df = pd.DataFrame(corr_rows)
thr_df = pd.DataFrame(thr_rows)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", None)
pd.set_option(
    "display.float_format",
    lambda x: f"{x:.3e}" if abs(x) < 1e-2 or abs(x) > 1e3 else f"{x:.3f}",
)

print("=== Correctness: Magi SP vs FA3-Ulysses (max abs / rel) ===")
print(corr_df.to_string(index=False))
print("\n=== Throughput (ms / TFLOP/s) ===")
cols = [
    "world_size",
    "mask",
    "dims",
    "magi_fwd_ms",
    "fa3_ulysses_fwd_ms",
    "magi_fwd_tflops",
    "fa3_ulysses_fwd_tflops",
    "magi_1f1b_ms",
    "fa3_ulysses_1f1b_ms",
    "magi_1f1b_tflops",
    "fa3_ulysses_1f1b_tflops",
]
print(thr_df[cols].to_string(index=False))

corr_csv = RESULTS / "correctness.csv"
thr_csv = RESULTS / "throughput.csv"
md_path = RESULTS / "magi_sp_vs_fa3_ulysses.md"
corr_df.to_csv(corr_csv, index=False, float_format="%.6e")
thr_df.to_csv(thr_csv, index=False, float_format="%.6e")

meta = payloads[WORLD_SIZES[0]]
md = [
    "# MagiAttention SP vs FA3-Ulysses",
    "",
    f"- seqlen={meta['seqlen']}, H={meta['num_heads']}, dtype={meta['dtype']}",
    f"- head_dims={meta['head_dims']}",
    f"- world_sizes={WORLD_SIZES}",
    "- relative error: max over entries with |ref|>1e-2",
    "",
    "## Correctness (Magi − FA3-Ulysses)",
    "",
    corr_df.to_markdown(index=False, floatfmt=".3e"),
    "",
    "## Throughput",
    "",
    thr_df[cols].to_markdown(index=False, floatfmt=".3f"),
    "",
]
md_path.write_text("\n".join(md), encoding="utf-8")
print("\nsaved:")
print(" ", corr_csv)
print(" ", thr_csv)
print(" ", md_path)



## Notes

- **Magi SP**: tokens are load-balanced along the sequence; attention + KV gather/scatter live inside `calc_attn`.
- **FA3-Ulysses**: each rank keeps a contiguous `seqlen/cp` shard with all heads, all2all to full sequence / `heads/cp`, runs FA3, all2all back. Requires `H % cp == 0`.
- Correctness compares Magi global `out/dq/dk/dv` to FA3-Ulysses gathered to the same global layout.
- Relative error is max over entries with `|ref| > 1e-2` (avoids blow-up on near-zeros in bf16).
- Throughput FLOPs: `2 * area * H * (dqk + dv)` fwd, `×2.5` bwd → TFLOP/s (asymmetric-aware, same as cutedsl notebook).
- Causal bwd (`dk`/`dv`) is typically noisier than full.
- For a deeper CP sweep (Ring / USP / more masks), see `exps/dist_attn/run_benchmark.sh`.

